In [ ]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = np.array(["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## TASK 1:CLEANING

In [ ]:
def clean_readings(raw):
    """Convert raw strings to int; impute invalid entries with the rounded mean of valid ones."""
    parsed, bad_idx = [], []
    for i, item in enumerate(raw):
        try:
            parsed.append(int(item))      # int(" 20 ") works: whitespace is stripped
        except ValueError:
            parsed.append(None)
            bad_idx.append(i)
    valid = [v for v in parsed if v is not None]
    fill = round(sum(valid) / len(valid))
    cleaned = [fill if v is None else v for v in parsed]
    return cleaned, bad_idx, fill

machakos_clean, bad_idx, fill = clean_readings(machakos_raw)
imputed_months = [months[i] for i in bad_idx]
print("Imputed months:", imputed_months)
print("Imputed with:", fill, "mm")
print("Cleaned Machakos:", machakos_clean)

Imputed months: [np.str_('April'), np.str_('September')]
Imputed with: 68 mm
Cleaned Machakos: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## TASK 2: COMBINE

In [ ]:
data = np.array(clean_rows + [machakos_clean])
print("Shape:", data.shape)
print("dtype:", data.dtype)

Shape: (5, 12)
dtype: int64


## TASK 3:TOTALS AND AVARAGE

In [ ]:
annual_totals = data.sum(axis=1)
monthly_avg = data.mean(axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")
print()
for m, avg in zip(months, monthly_avg):
    print(f"{m}: {avg:.1f} mm")

Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## TASK 4:WETTEST MONTH

In [ ]:
Wettest_idx = np.argmax(data, axis=1)
Wettest_names = months[Wettest_idx]
for county, month in zip(counties, Wettest_names):
    print(f"{county}: {month}")

Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## TASK 5:DRY MONTHS

In [ ]:
dry_mask = data < 50
dry_counts = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_counts):
    print(f"{county}: {n} dry months")

Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


## TASK 6 :PLANTING  WINDOWS


In [ ]:
pair_sums = data[:, :-1] + data[:, 1:]       # shape (5, 11): Jan+Feb, Feb+Mar, ...
best_start = np.argmax(pair_sums, axis=1)

windows = [f"{months[s][:3]}-{months[s + 1][:3]}" for s in best_start]
for county, w, total in zip(counties, windows, pair_sums.max(axis=1)):
    print(f"{county}: {w} ({total} mm)")

Nairobi: Apr-May (395 mm)
Nakuru: Apr-May (265 mm)
Kisumu: Apr-May (420 mm)
Mombasa: Apr-May (390 mm)
Machakos: Nov-Dec (278 mm)


##TASK7:CSV

In [ ]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for row in zip(counties, annual_totals, Wettest_names, dry_counts, windows):
        writer.writerow([row[0], int(row[1]), row[2], int(row[3]), row[4]])

print(open("rainfall_summary.csv").read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,Apr-May
Nakuru,916,April,3,Apr-May
Kisumu,1371,April,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,November,6,Nov-Dec



## Task 8 (markdown): Planting Advisory

In [ ]:
## Planting Advisory

Nairobi, Nakuru, Kisumu and Mombasa all receive their strongest two-month rains in April–May, so farmers there should prepare land in March and plant as the long rains begin. Machakos is the exception: its best window is November–December, and it is also the driest county with six months below 50 mm, so farmers should favour drought-tolerant crops and moisture-saving practices. Kisumu is the most reliable county (1,371 mm and no dry months), while Nairobi has four dry months from June to September, so late planting there is risky. Because only one year of data was used, SMS advice should be checked against the seasonal forecast before it is sent.

##Task 9 (markdown):Data Quality Notes

In [ ]:
## Data Quality Notes

- Two Machakos readings (April "N/A" and September blank) were invalid and were replaced with the rounded average of the ten valid months, 68 mm.
- Machakos' annual total (811 mm) is therefore partly an estimate. Its dry-month count may also be understated, because the true September reading could have been below 50 mm.
- The Machakos Nov–Dec window uses only real readings, so it is the most trustworthy result for that county. The other four counties' data was treated as clean and was not independently verified.

## txt

In [ ]:
with open("requirements.txt", "w") as f:
    f.write("numpy\n")